### 這個colab檔案負責處理斷詞，在處理訓練資料(已完成)和處理使用者輸入資料都會用到

In [19]:
!pip install -U ckip-transformers

In [20]:
import gdown
import os
import glob
import re
import pandas as pd
import shutil # For copying files

# ==========================================
# 1. 下載 GitHub 資料夾中的所有 .txt 檔案
# ==========================================
folder_url = 'https://github.com/Joanne334/Intro-to-AI-g8/tree/5e3e2e59e87941219f09850b63bd3c8c6dc46ded/txt_datas/texts/*'

# Extract repository information from the folder_url
match = re.match(r'(https://github.com/[^/]+/[^/]+)/tree/[^/]+/(.*)/\*', folder_url)
if not match:
    raise ValueError("Invalid GitHub folder URL format provided.")

repo_base_url = match.group(1)
source_folder_relative_path = match.group(2)
repo_name = repo_base_url.split('/')[-1] # e.g., Intro-to-AI-g8
repo_url = repo_base_url + '.git'

# Define output folder name
output_dir = 'downloaded_files'

# Check and create output folder
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"已創建輸出資料夾: {output_dir}")
else:
    # Clear existing content in output_dir to ensure fresh download
    for item in os.listdir(output_dir):
        item_path = os.path.join(output_dir, item)
        if os.path.isfile(item_path) or os.path.islink(item_path):
            os.unlink(item_path)
        elif os.path.isdir(item_path):
            shutil.rmtree(item_path)
    print(f"已清空現有輸出資料夾: {output_dir}")


# Clean up existing cloned repository if it exists from previous runs
if os.path.exists(repo_name):
    shutil.rmtree(repo_name)
    print(f"已移除舊的克隆儲存庫資料夾: {repo_name}")


# Clone the GitHub repository
print(f"正在從 {repo_url} 克隆儲存庫...")
!git clone "{repo_url}"
print(f"儲存庫 {repo_name} 克隆完成。")

# Define the source path for the .txt files within the cloned repository
source_folder_in_repo = os.path.join(repo_name, source_folder_relative_path)

# Check if the source folder exists after cloning
if os.path.exists(source_folder_in_repo):
    print(f"正在從 '{source_folder_in_repo}' 複製 .txt 檔案到 '{output_dir}'...")
    # Get all .txt files from the source folder
    txt_files_to_copy = glob.glob(os.path.join(source_folder_in_repo, "*.txt"))

    if not txt_files_to_copy:
        print(f"警告：在 {source_folder_in_repo} 中找不到任何 .txt 檔案。")
    else:
        for file_path in txt_files_to_copy:
            shutil.copy(file_path, output_dir)
            print(f"已複製檔案: {os.path.basename(file_path)}")
        print(f"所有 .txt 檔案已複製到: {output_dir}")
else:
    print(f"錯誤：在克隆的儲存庫中找不到資料夾 '{source_folder_in_repo}'。請檢查 '{source_folder_relative_path}' 路徑是否正確。")

# Clean up the cloned repository folder to avoid clutter
if os.path.exists(repo_name):
    shutil.rmtree(repo_name)
    print(f"已移除克隆的儲存庫資料夾: {repo_name}")

print(f"\n資料夾內容已下載至: {output_dir}")

已清空現有輸出資料夾: downloaded_files
正在從 https://github.com/Joanne334/Intro-to-AI-g8.git 克隆儲存庫...
Cloning into 'Intro-to-AI-g8'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (147/147), done.
remote: Total 152 (delta 12), reused 5 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 110.73 KiB | 1.73 MiB/s, done.
Resolving deltas: 100% (12/12), done.
儲存庫 Intro-to-AI-g8 克隆完成。
正在從 'Intro-to-AI-g8/txt_datas/texts' 複製 .txt 檔案到 'downloaded_files'...
已複製檔案: 用膝蓋跳舞的女孩.txt
已複製檔案: 篇目3身在此山中王昶雄.txt
已複製檔案: 篇目1滿山花兒開林良.txt
已複製檔案: 拔不起來的筆.txt
已複製檔案: 雞蛋糕.txt
已複製檔案: 養雞 林良.txt
已複製檔案: 畫龍點睛.txt
已複製檔案: 愛和信心強過成績單陳美儒.txt
已複製檔案: 篇目2等待一個站起來的餃子琦君.txt
已複製檔案: 大腸包小腸.txt
已複製檔案: 石虎兄妹.txt
已複製檔案: 山與海的交響樂──東海岸鐵道.txt
已複製檔案: 元宵節.txt
已複製檔案: 小紅帽.txt
已複製檔案: 童稚的世界王慕真.txt
已複製檔案: 士林夜市.txt
已複製檔案: 三、阿嬤與歌仔戲.txt
已複製檔案: 牛肉麵.txt
已複製檔案: 棒球英雄夢.txt
已複製檔案: 春節.txt
已複製檔案: 聰明的小兔子.txt
已複製檔案: 神驢辦案.txt
已複製檔案: 篇目3小太陽林良.txt
已複製檔案: 篇目4十月桂花香余麗珠.txt
已複製檔案: 做泡菜.txt


In [21]:
# ==========================================
# 2. 定義切片與資料夾處理邏輯
# ==========================================
def process_all_txt_in_folder(directory_path, max_length=200):
    all_segments_data = []

    # 遞迴尋找資料夾內所有的 .txt 檔案
    txt_files = glob.glob(os.path.join(directory_path, "**/*.txt"), recursive=True)

    if not txt_files:
        print("警告：在指定的資料夾中找不到任何 .txt 檔案。")
        return pd.DataFrame()

    print(f"找到 {len(txt_files)} 個 txt 檔案，開始進行切片處理...")

    for file_path in txt_files:
        # 讀取單一文本
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                lines = f.readlines()
        except UnicodeDecodeError:
            print(f"警告：檔案 {file_path} 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。")
            try:
                with open(file_path, 'r', encoding='big5') as f:
                    lines = f.readlines()
            except UnicodeDecodeError as e:
                print(f"錯誤：檔案 {file_path} 無法以 BIG5 編碼讀取，跳過此檔案。錯誤訊息: {e}")
                continue # Skip this file if it can't be decoded with BIG5 either
        except Exception as e:
            print(f"讀取檔案 {file_path} 時發生未知錯誤，跳過此檔案。錯誤訊息: {e}")
            continue

        if not lines:
            continue

        content = "".join(lines)

        # 切片邏輯 (以換行符號切割成段落)
        paragraphs = [p.strip() for p in content.split('\n') if p.strip()]

        segments = []
        for p in paragraphs:
            # 若段落長度在限制內，直接算作一個片段
            if len(p) <= max_length:
                segments.append(p)
            else:
                # 若段落過長，按標點符號進行「軟切分」
                sentences = re.split(r'([。！？；])', p)
                current_segment = ""

                for i in range(0, len(sentences) - 1, 2):
                    sentence = sentences[i]
                    punctuation = sentences[i+1] if (i+1) < len(sentences) else ""
                    full_sentence = sentence + punctuation

                    if len(current_segment) + len(full_sentence) <= max_length:
                        current_segment += full_sentence
                    else:
                        if current_segment:
                            segments.append(current_segment)
                        current_segment = full_sentence

                if current_segment:
                    segments.append(current_segment)

        # 將該檔案切出來的所有片段，加入總清單中
        for i, seg in enumerate(segments):
            all_segments_data.append({
                "source_file": os.path.basename(file_path), # 紀錄來源檔名以便追蹤
                "segment_id": i + 1,
                "text_segment": seg,
                "length": len(seg)
            })

    # 將所有資料轉換為 DataFrame
    return pd.DataFrame(all_segments_data)



# ==========================================
# 3. 執行處理並匯出結果
# ==========================================
# 執行批次處理 (設定 max_length=200)
df_all_segments = process_all_txt_in_folder(output_dir, max_length=200)

if not df_all_segments.empty:
    print(f"\n處理完成！共產出 {len(df_all_segments)} 個文本片段。")

    # 匯出成 CSV 檔案供後續 BERT 訓練使用
    csv_filename = "processed_segments.csv"
    df_all_segments.to_csv(csv_filename, index=False, encoding='utf-8-sig') # 使用 utf-8-sig 避免 Excel 開啟亂碼
    print(f"已將結果儲存為 {csv_filename}")

    # 顯示前 5 筆結果預覽
    display(df_all_segments.head())

找到 76 個 txt 檔案，開始進行切片處理...
警告：檔案 downloaded_files/用膝蓋跳舞的女孩.txt 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。
警告：檔案 downloaded_files/雞蛋糕.txt 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。
警告：檔案 downloaded_files/大腸包小腸.txt 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。
警告：檔案 downloaded_files/元宵節.txt 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。
警告：檔案 downloaded_files/士林夜市.txt 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。
警告：檔案 downloaded_files/牛肉麵.txt 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。
警告：檔案 downloaded_files/春節.txt 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。
警告：檔案 downloaded_files/芋圓.txt 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。
警告：檔案 downloaded_files/擔仔麵.txt 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。
警告：檔案 downloaded_files/鼎邊銼.txt 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。
錯誤：檔案 downloaded_files/鼎邊銼.txt 無法以 BIG5 編碼讀取，跳過此檔案。錯誤訊息: 'big5' codec can't decode byte 0xff in position 239: illegal multibyte sequence
警告：檔案 downloaded_files/陽明山國家公園.txt 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。
警告：檔案 downloaded_files/蚵仔煎.txt 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。
警告：檔案 downloaded_files/兒童節、清明節.txt 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。
警告：檔案 downloaded_files/臭豆腐.txt 無法以 UTF-8 編

,source_file,segment_id,text_segment,length
0,用膝蓋跳舞的女孩.txt,1,欣賞了一場特別的表演後，我們全家人都感動得說不出話來。舞臺上的女孩沒有腳，卻能跳出美妙的舞姿...,83
1,用膝蓋跳舞的女孩.txt,2,這位女孩名叫「郭韋齊」，從小就喜歡唱唱跳跳，自得其樂。在七歲那年，因為不明原因的疾病，引發休...,172
2,用膝蓋跳舞的女孩.txt,3,不過，韋齊的心裡，始終記得以前跳舞、彈琴的快樂時光，於是在父母的支持下，重新再學習。別人穿鞋...,183
3,篇目3身在此山中王昶雄.txt,1,清晨，推開窗扉，清新之氣撲人眉宇，陽光躍然而入，滿室生輝。有了窗，一間容膝斗室，便變成頂天立...,186
4,篇目3身在此山中王昶雄.txt,2,陽明、大屯兩瀑布匯流出谷，谷口是石壇路峰頂橋，橋畔有洞形月門及花廊，景色幽美。回到小屋，已是...,164


In [22]:
# ==========================================
# 步驟二：載入 CKIP 斷詞 (WS) 與詞性標記 (POS) 模型
# ==========================================
from ckip_transformers.nlp import CkipWordSegmenter, CkipPosTagger

print("正在載入 CKIP 模型，這可能需要幾分鐘的時間（會自動下載模型檔）...")
# 參數說明：
# model="albert-base" 是一個在效能與速度上達到很好平衡的預訓練模型。
# device=0 代表使用 Colab 的 GPU 進行運算加速。若未來在沒有 GPU 的環境執行，請改為 device=-1。
ws_driver = CkipWordSegmenter(model="albert-base", device=0)
pos_driver = CkipPosTagger(model="albert-base", device=0)
print("模型載入完成！")


正在載入 CKIP 模型，這可能需要幾分鐘的時間（會自動下載模型檔）...


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

模型載入完成！


In [23]:
# ==========================================
# 步驟三：實作「批次斷詞」邏輯 (Batch Processing)
# ==========================================
# 這裡會直接使用上一步產生的 df_all_segments
if not df_all_segments.empty:
    print(f"\n準備對 {len(df_all_segments)} 個文本片段進行批次處理...")

    # 將 DataFrame 中的 text_segment 欄位抽出來變成一維的 Python List
    text_list = df_all_segments['text_segment'].tolist()

    # 將整個 List 一次性餵給模型。
    # ckip-transformers 底層有很好的 batch 處理機制，一次丟整個 List 的運算效率遠高於寫 for 迴圈逐句處理。
    print("1. 正在執行斷詞 (WS)...")
    ws_results = ws_driver(text_list)

    print("2. 正在執行詞性標記 (POS)...")
    # POS 模型需要以斷詞的結果作為輸入
    pos_results = pos_driver(ws_results)

    print("批次處理完成！這兩個變數 (ws_results, pos_results) 已經儲存了所有的運算結果。")
else:
    print("錯誤：找不到資料！請確認前一步驟的 df_all_segments 變數是否存在且有內容。")


準備對 569 個文本片段進行批次處理...
1. 正在執行斷詞 (WS)...


Inference: 100%|██████████| 3/3 [00:05<00:00,  1.86s/it]


2. 正在執行詞性標記 (POS)...


Inference: 100%|██████████| 19/19 [00:20<00:00,  1.10s/it]

批次處理完成！這兩個變數 (ws_results, pos_results) 已經儲存了所有的運算結果。


In [24]:
# ==========================================
# 步驟四：將結果寫回並清理 DataFrame
# ==========================================
print("正在將 CKIP 運算結果整合回 DataFrame...")

# 確保運算結果的長度與原始資料表的列數一致，避免對齊錯誤
if len(ws_results) == len(df_all_segments) and len(pos_results) == len(df_all_segments):

    # 1. 寫入原始的 List 格式（方便後續程式邏輯讀取與比對）
    df_all_segments['ckip_ws'] = ws_results
    df_all_segments['ckip_pos'] = pos_results

    # 2. 新增一欄「空白鍵連接的字串格式」
    # 這是為了方便你之後寫 Regex (正規表達式) 或是肉眼檢查
    # x 是一個 list，例如 ['今天', '天氣', '好']，會被轉成 "今天 天氣 好"
    df_all_segments['ckip_ws_joined'] = df_all_segments['ckip_ws'].apply(lambda x: " ".join(x))

    # 3. 同理，把詞性也串起來方便檢查
    df_all_segments['ckip_pos_joined'] = df_all_segments['ckip_pos'].apply(lambda x: " ".join(x))

    print("資料整合成功！")
else:
    print("錯誤：CKIP 輸出結果的長度與 DataFrame 筆數不符合，請確認前面的步驟是否有中斷。")




正在將 CKIP 運算結果整合回 DataFrame...
資料整合成功！


In [25]:
# ==========================================
# 步驟五：匯出最終版本的 CSV
# ==========================================
# 定義新的輸出檔名，避免覆蓋原本還沒斷詞的檔案
final_csv_filename = "processed_segments_with_ckip_story.csv"

# 匯出 CSV，使用 utf-8-sig 確保在 Windows Excel 打開不會亂碼
df_all_segments.to_csv(final_csv_filename, index=False, encoding='utf-8-sig')

print(f"\n大功告成！已將帶有 CKIP 斷詞特徵的資料儲存為：{final_csv_filename}")

# 顯示前 5 筆結果預覽，檢查欄位是否都有正確長出來
display(df_all_segments[['text_segment', 'ckip_ws', 'ckip_pos', 'ckip_ws_joined']].head())


大功告成！已將帶有 CKIP 斷詞特徵的資料儲存為：processed_segments_with_ckip_story.csv


,text_segment,ckip_ws,ckip_pos,ckip_ws_joined
0,欣賞了一場特別的表演後，我們全家人都感動得說不出話來。舞臺上的女孩沒有腳，卻能跳出美妙的舞姿...,"[欣賞, 了, 一, 場, 特別, 的, 表演, 後, ，, 我們, 全, 家人, 都, 感...","[VJ, Di, Neu, Nf, VH, DE, Na, Ng, COMMACATEGOR...",欣賞 了 一 場 特別 的 表演 後 ， 我們 全 家人 都 感動 得 說 不 出 話 來 ...
1,這位女孩名叫「郭韋齊」，從小就喜歡唱唱跳跳，自得其樂。在七歲那年，因為不明原因的疾病，引發休...,"[這, 位, 女孩, 名叫, 「, 郭韋齊, 」, ，, 從小, 就, 喜歡, 唱唱, 跳跳...","[Nep, Nf, Na, VG, PARENTHESISCATEGORY, Nb, PAR...",這 位 女孩 名叫 「 郭韋齊 」 ， 從小 就 喜歡 唱唱 跳跳 ， 自得其樂 。 在 七...
2,不過，韋齊的心裡，始終記得以前跳舞、彈琴的快樂時光，於是在父母的支持下，重新再學習。別人穿鞋...,"[不過, ，, 韋齊, 的, 心, 裡, ，, 始終, 記得, 以前, 跳舞, 、, 彈琴,...","[Cbb, COMMACATEGORY, Nb, DE, Na, Ng, COMMACATE...",不過 ， 韋齊 的 心 裡 ， 始終 記得 以前 跳舞 、 彈琴 的 快樂 時光 ， 於是 ...
3,清晨，推開窗扉，清新之氣撲人眉宇，陽光躍然而入，滿室生輝。有了窗，一間容膝斗室，便變成頂天立...,"[清晨, ，, 推開, 窗扉, ，, 清新, 之, 氣, 撲, 人, 眉宇, ，, 陽光, ...","[Nd, COMMACATEGORY, VC, Na, COMMACATEGORY, VH,...",清晨 ， 推開 窗扉 ， 清新 之 氣 撲 人 眉宇 ， 陽光 躍然 而 入 ， 滿 室 生...
4,陽明、大屯兩瀑布匯流出谷，谷口是石壇路峰頂橋，橋畔有洞形月門及花廊，景色幽美。回到小屋，已是...,"[陽明, 、, 大屯, 兩, 瀑布, 匯流, 出, 谷, ，, 谷口, 是, 石壇路, 峰頂...","[Nb, PAUSECATEGORY, Nc, Neu, Na, VH, VC, Na, C...",陽明 、 大屯 兩 瀑布 匯流 出 谷 ， 谷口 是 石壇路 峰頂橋 ， 橋 畔 有 洞形 ...
